In [1]:
import pandas as pd
import yaml
import random
import json
from collections import defaultdict

In [2]:
# =============================
# CONFIG
# =============================
INPUT_FILE = "7817_1.csv"
OUTPUT_PRODUCTS = "products.yml"
OUTPUT_USERS = "users.yml"
OUTPUT_REVIEWS = "reviews.yml"

In [3]:
# =============================
# LOAD DATA
# =============================
df = pd.read_csv(INPUT_FILE)

In [4]:
# =============================
# HELPERS
# =============================

def generate_description(row):
    """
    Genera una descripción tipo marketing basada en los campos del producto.
    Simula un prompt.
    """
    name = row.get("name", "")
    brand = row.get("brand", "")
    category = row.get("categories", "")
    colors = row.get("colors", "")
    dimension = row.get("dimension", "")
    weight = row.get("weight", "")

    prompt = f"""
    Write a compelling ecommerce product description for:
    Product: {name}
    Brand: {brand}
    Category: {category}
    Colors: {colors}
    Dimensions: {dimension}
    Weight: {weight}
    Highlight quality, usability and value.
    """

    # Simulación de respuesta generada
    description = (
        f"{name} by {brand} is a premium {category}. "
        f"Designed with attention to detail and available in {colors}. "
        f"Its dimensions ({dimension}) and weight ({weight}) make it practical and versatile. "
        f"Perfect for customers looking for quality and reliability."
    )

    return description.strip()


def extract_price(price_field):
    try:
        if isinstance(price_field, str) and price_field.startswith("{"):
            price_json = json.loads(price_field)
            return float(price_json.get("amountMax", 0))
        return float(price_field)
    except:
        return 0.0

In [5]:
# =============================
# BUILD UNIQUE PRODUCTS
# =============================

unique_products = df.drop_duplicates(subset=["id"])

products_list = []

for _, row in unique_products.iterrows():
    product = {
        "id": row["id"],  # recomendable mantenerlo
        "name": row.get("name"),
        "description": generate_description(row),
        "price": extract_price(row.get("prices")),
        "isActive": True,
        "category": row.get("categories"),
        "stock": random.randint(5, 100),
        "images": [
            f"https://dummyimage.com/600x400/000/fff&text={row.get('name')}"
        ]
    }

    products_list.append(product)

In [6]:
# =============================
# BUILD USERS
# =============================
users_dict = {}

for _, row in df.iterrows():
    username = row.get("reviews.username")

    if pd.notna(username) and username not in users_dict:
        users_dict[username] = {
            "username": username,
            "city": row.get("reviews.userCity"),
            "province": row.get("reviews.userProvince"),
        }

In [7]:
# =============================
# BUILD REVIEWS
# =============================
reviews_list = []

for _, row in df.iterrows():
    if pd.notna(row.get("reviews.text")):
        reviews_list.append({
            "product_id": row.get("id"),
            "username": row.get("reviews.username"),
            "rating": row.get("reviews.rating"),
            "title": row.get("reviews.title"),
            "comment": row.get("reviews.text"),
            "date": row.get("reviews.date"),
            "recommended": row.get("reviews.doRecommend")
        })

In [9]:
# =============================
# SAVE YAML FILES
# =============================
with open(OUTPUT_PRODUCTS, "w", encoding="utf-8") as f:
    yaml.dump(products_list, f, allow_unicode=True)

with open(OUTPUT_USERS, "w", encoding="utf-8") as f:
    yaml.dump(list(users_dict.values()), f, allow_unicode=True)

with open(OUTPUT_REVIEWS, "w", encoding="utf-8") as f:
    yaml.dump(reviews_list, f, allow_unicode=True)

print("✅ YAML files generated successfully!")

✅ YAML files generated successfully!


In [3]:
raw_dv = pd.read_csv("7817_1.csv")
raw_dv.sample(5)

,id,asins,brand,categories,colors,dateAdded,dateUpdated,dimension,ean,keys,...,reviews.rating,reviews.sourceURLs,reviews.text,reviews.title,reviews.userCity,reviews.userProvince,reviews.username,sizes,upc,weight
792,AVpe7LD5LJeJML43ybWA,"B00DOPNO4M,B00BWYQ9YE,B00CYQPMJC,B00CUU1CGY,B0...",Amazon,"Amazon Devices,Kindle Store,buy a kindle",NaN,2015-05-22T15:33:59Z,2017-07-18T23:52:40Z,NaN,NaN,"kindlefirehdx7/b00dopno4m,kindlefirehdx7/b00bw...",...,NaN,http://www.amazon.com/Kindle-Fire-HDX-Display-...,This is the middle model of the three models t...,Excellent 3rd-generation tablet that compares ...,NaN,NaN,NF,NaN,NaN,NaN
1069,AVpfpK8KLJeJML43BCuD,B01BH83OOM,Amazon,"Amazon Devices,Home,Smart Home & Connected Liv...",Black,2017-01-04T03:51:17Z,2017-08-13T08:31:07Z,4.8 in x 6.6 in x 3.2 in,8.416670e+11,amazontapalexaenabledportablebluetoothspeaker/...,...,5.0,http://reviews.bestbuy.com/3545/5097300/review...,Great sound and functionality. Minimal kinks i...,Awesome for college students,NaN,NaN,Supahfree,NaN,8.416670e+11,1.75 lbs
1527,AVpge-anilAPnD_xtDVf,B00HX0SRXW,Amazon,"Amazon Devices,Corded Headsets,Electronics Fea...",Black,2015-07-13T14:50:00Z,2017-08-13T08:29:03Z,NaN,8.487190e+11,"amazonpremiumheadphones/b00hx0srxw,08487190395...",...,3.0,http://www.amazon.com/Amazon-KA416Y-Premium-He...,While I've purchased items from Amazon for yea...,Awesome Headphones! Longevity An Issue 798 peo...,NaN,NaN,A. Younan,NaN,8.487190e+11,NaN
1560,AVpge-anilAPnD_xtDVf,B00HX0SRXW,Amazon,"Amazon Devices,Corded Headsets,Electronics Fea...",Black,2015-07-13T14:50:00Z,2017-08-13T08:29:03Z,NaN,8.487190e+11,"amazonpremiumheadphones/b00hx0srxw,08487190395...",...,5.0,http://www.amazon.com/Amazon-KA416Y-Premium-He...,"I bought these for a couple of reasons.First, ...",I hate having to shove headphones into my brai...,NaN,NaN,Andrew,NaN,8.487190e+11,NaN
5,AVpe7AsMilAPnD_xQ78G,B00QJDU3KY,Amazon,"Amazon Devices,mazon.co.uk",NaN,2016-03-08T20:21:53Z,2017-07-18T23:52:58Z,169 mm x 117 mm x 9.1 mm,NaN,kindlepaperwhite/b00qjdu3ky,...,NaN,http://www.amazon.com/Kindle-Paperwhite-High-R...,"My previous kindle was a DX, this is my second...",Great device for reading. 8 people found this ...,NaN,NaN,Kelvin Law,NaN,NaN,205 grams


In [4]:
raw_dv.columns

Index(['id', 'asins', 'brand', 'categories', 'colors', 'dateAdded',
       'dateUpdated', 'dimension', 'ean', 'keys', 'manufacturer',
       'manufacturerNumber', 'name', 'prices', 'reviews.date',
       'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating',
       'reviews.sourceURLs', 'reviews.text', 'reviews.title',
       'reviews.userCity', 'reviews.userProvince', 'reviews.username', 'sizes',
       'upc', 'weight'],
      dtype='object')

In [7]:
raw_dv["id"].unique().shape

(66,)

In [15]:
raw_dv["name"].unique()

array(['Kindle Paperwhite', 'Kindle Keyboard',
       'Certified Refurbished Amazon Fire TV (Previous Generation - 1st)',
       'Amazon Echo Dot Case (fits Echo Dot 2nd Generation only) - Indigo Fabric',
       'Amazon Tap Sling Cover - Tangerine', 'Kindle Fire HDX 8.9"',
       'Amazon Echo Dot Case (fits Echo Dot 2nd Generation only) - Saddle Tan Leather',
       'Amazon Tap Sling Cover - Blue',
       'Amazon Echo Dot Case (fits Echo Dot 2nd Generation only) - Sandstone Fabric',
       'All-New Amazon Fire TV Game Controller', 'Fire HD 7 Tablet',
       'Amazon Fire TV Game Controller', 'Amazon Tap Sling Cover - White',
       'Amazon Tap Sling Cover - Magenta',
       'Certified Refurbished Kindle E-reader',
       'Amazon Tap Sling Cover - Green',
       'Kindle Paperwhite E-reader - Black', 'Kindle Paperwhite 3G',
       'Fire HD 10 Tablet with Alexa',
       'Amazon Echo Dot Case (fits Echo Dot 2nd Generation only) - Merlot Leather',
       'Fire Kids Edition Tablet', 'Echo Sho